## Experiment No: 9
## Experiment Title: Text Summarization using Transformer Models

**Name:** Himanshu Jadhav  
**Roll Number:** TE-32

### Step 1: Import Libraries

In [1]:
import nltk
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from tabulate import tabulate

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

### Step 2: Sample Long-Form Article

In [2]:
article = """
Artificial intelligence is transforming the way businesses operate across the world.
Companies are increasingly using machine learning models to automate repetitive tasks
and improve decision making. Natural language processing, a branch of AI, allows
computers to understand and generate human language. This has led to major advances
in chatbots, translation systems, and text summarization tools. However, experts warn
that AI systems can also introduce bias if the training data is not carefully selected.
Researchers are working on techniques to make AI models more fair and transparent.
Governments around the world are also starting to introduce regulations to ensure AI
is used responsibly. Despite the challenges, AI adoption continues to grow rapidly
across industries such as healthcare, finance, and education. Many experts believe
that the next decade will see even faster progress in AI capabilities.
"""

reference_summary = ("AI is transforming businesses through automation and NLP advances, "
                      "but bias and regulation remain challenges as adoption grows across industries.")

print(article)
print("\nReference Summary:")
print(reference_summary)


Artificial intelligence is transforming the way businesses operate across the world.
Companies are increasingly using machine learning models to automate repetitive tasks
and improve decision making. Natural language processing, a branch of AI, allows
computers to understand and generate human language. This has led to major advances
in chatbots, translation systems, and text summarization tools. However, experts warn
that AI systems can also introduce bias if the training data is not carefully selected.
Researchers are working on techniques to make AI models more fair and transparent.
Governments around the world are also starting to introduce regulations to ensure AI
is used responsibly. Despite the challenges, AI adoption continues to grow rapidly
across industries such as healthcare, finance, and education. Many experts believe
that the next decade will see even faster progress in AI capabilities.


Reference Summary:
AI is transforming businesses through automation and NLP advanc

### Step 3: Extractive Summarization (TF-IDF Sentence Scoring)

In [3]:
cleaned_article = ' '.join(article.split())
sentences = sent_tokenize(cleaned_article)
print('Number of sentences:', len(sentences))

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)

# Score each sentence by summing its TF-IDF weights
sentence_scores = tfidf_matrix.sum(axis=1).A1

# Pick top 3 sentences, keeping their original order
top_indices = np.argsort(sentence_scores)[-3:]
top_indices_sorted = sorted(top_indices)

extractive_summary = ' '.join([sentences[i] for i in top_indices_sorted])

print('Sentence Scores:')
for i, score in enumerate(sentence_scores):
    print(f'[{i}] {score:.3f} - {sentences[i][:60]}...')

print('\nExtractive Summary:')
print(extractive_summary)

Number of sentences: 9
Sentence Scores:
[0] 2.642 - Artificial intelligence is transforming the way businesses o...
[1] 3.461 - Companies are increasingly using machine learning models to ...
[2] 3.003 - Natural language processing, a branch of AI, allows computer...
[3] 2.996 - This has led to major advances in chatbots, translation syst...
[4] 3.122 - However, experts warn that AI systems can also introduce bia...
[5] 2.787 - Researchers are working on techniques to make AI models more...
[6] 2.959 - Governments around the world are also starting to introduce ...
[7] 3.283 - Despite the challenges, AI adoption continues to grow rapidl...
[8] 2.602 - Many experts believe that the next decade will see even fast...

Extractive Summary:
Companies are increasingly using machine learning models to automate repetitive tasks and improve decision making. However, experts warn that AI systems can also introduce bias if the training data is not carefully selected. Despite the challenges, AI ado

### Step 4: Load Pretrained Transformer Summarizer (DistilBART)

In [4]:
model_name = 'sshleifer/distilbart-cnn-12-6'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print('Model loaded successfully:', model_name)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded successfully: sshleifer/distilbart-cnn-12-6


### Step 5: Generate Abstractive Summary

In [5]:
inputs = tokenizer(cleaned_article, return_tensors='pt', truncation=True)
summary_ids = model.generate(
    **inputs,
    max_length=60,
    min_length=20,
    num_beams=4,
    do_sample=False,
    forced_bos_token_id=0
)
abstractive_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print('Abstractive Summary:')
print(abstractive_summary)

Abstractive Summary:
 Companies are increasingly using machine learning models to automate repetitive tasks . Natural language processing, a branch of AI, allows computers to understand and generate human language . This has led to major advances in chatbots, translation systems and text summarization tools .


### Step 6: Compare Both Summaries with the Reference

In [6]:
print("Reference Summary  :", reference_summary)
print()
print("Extractive Summary :", extractive_summary)
print()
print("Abstractive Summary:", abstractive_summary)

Reference Summary  : AI is transforming businesses through automation and NLP advances, but bias and regulation remain challenges as adoption grows across industries.

Extractive Summary : Companies are increasingly using machine learning models to automate repetitive tasks and improve decision making. However, experts warn that AI systems can also introduce bias if the training data is not carefully selected. Despite the challenges, AI adoption continues to grow rapidly across industries such as healthcare, finance, and education.

Abstractive Summary:  Companies are increasingly using machine learning models to automate repetitive tasks . Natural language processing, a branch of AI, allows computers to understand and generate human language . This has led to major advances in chatbots, translation systems and text summarization tools .


### Step 7: Evaluate using ROUGE Scores

In [7]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

extractive_scores = scorer.score(reference_summary, extractive_summary)
abstractive_scores = scorer.score(reference_summary, abstractive_summary)

print("Extractive Summary ROUGE Scores:")
for metric, score in extractive_scores.items():
    print(f"  {metric}: precision={score.precision:.3f}, recall={score.recall:.3f}, f1={score.fmeasure:.3f}")

print("\nAbstractive Summary ROUGE Scores:")
for metric, score in abstractive_scores.items():
    print(f"  {metric}: precision={score.precision:.3f}, recall={score.recall:.3f}, f1={score.fmeasure:.3f}")

Extractive Summary ROUGE Scores:
  rouge1: precision=0.240, recall=0.600, f1=0.343
  rouge2: precision=0.020, recall=0.053, f1=0.029
  rougeL: precision=0.160, recall=0.400, f1=0.229

Abstractive Summary ROUGE Scores:
  rouge1: precision=0.125, recall=0.250, f1=0.167
  rouge2: precision=0.000, recall=0.000, f1=0.000
  rougeL: precision=0.100, recall=0.200, f1=0.133


### Step 8: ROUGE Comparison Table

In [8]:
rouge_table = pd.DataFrame({
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Extractive F1': [
        round(extractive_scores['rouge1'].fmeasure, 3),
        round(extractive_scores['rouge2'].fmeasure, 3),
        round(extractive_scores['rougeL'].fmeasure, 3)
    ],
    'Abstractive F1': [
        round(abstractive_scores['rouge1'].fmeasure, 3),
        round(abstractive_scores['rouge2'].fmeasure, 3),
        round(abstractive_scores['rougeL'].fmeasure, 3)
    ]
})

print(tabulate(rouge_table, headers='keys', tablefmt='psql'))

+----+----------+-----------------+------------------+
|    | Metric   |   Extractive F1 |   Abstractive F1 |
|----+----------+-----------------+------------------|
|  0 | ROUGE-1  |           0.343 |            0.167 |
|  1 | ROUGE-2  |           0.029 |            0     |
|  2 | ROUGE-L  |           0.229 |            0.133 |
+----+----------+-----------------+------------------+


### Final Output

In [9]:
print("Experiment Completed Successfully")
print()
print(tabulate(rouge_table, headers='keys', tablefmt='fancy_grid'))

Experiment Completed Successfully

╒════╤══════════╤═════════════════╤══════════════════╕
│    │ Metric   │   Extractive F1 │   Abstractive F1 │
╞════╪══════════╪═════════════════╪══════════════════╡
│  0 │ ROUGE-1  │           0.343 │            0.167 │
├────┼──────────┼─────────────────┼──────────────────┤
│  1 │ ROUGE-2  │           0.029 │            0     │
├────┼──────────┼─────────────────┼──────────────────┤
│  2 │ ROUGE-L  │           0.229 │            0.133 │
╘════╧══════════╧═════════════════╧══════════════════╛
